In [ ]:
# Load packages
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import numpy as np 
import os 
import ast
import re

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# List of genes of interest (Ensembl gene IDs)
gene = ['ENSG00000119772', 
        'ENSG00000168769', 
        'ENSG00000182054', 
        'ENSG00000161547']

# Assign GCS workspace bucket
bucket = os.environ["WORKSPACE_BUCKET"]

# Extract variant annotations for genes of interest
def extract(df, gene): 
    '''
    Extract variant-level annotations from a SAIGE-GENE+ table.
    df   : SAIGE-GENE+ annotation table
    gene : List of Ensembl gene IDs
    '''
    df = df[df[0].str.contains("|".join(gene), na=False)]
    df = df[0].str.split(' ', expand=True)
    df = df.drop(columns=0).T
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    return df 

# Initialize lists to store variant IDs and annotations
ids, annotations = [], []

# Iterate over chromosomes containing genes of interest
for chrom in [2, 4, 15, 17]:
    
    # Define GCS and local file paths
    gcs_path = (
        f"{bucket}/data/annotation_table/"
        f"EUR_HC_annotation/HC-annotation-chr{chrom}.txt"
    )
    local_path = f"HC-annotation-chr{chrom}.txt"

    # Download chromosome-specific annotation table
    subprocess.run(
        ["gcloud", "storage", "cp", gcs_path, local_path],
        check=True
    )

    # Load and extract annotations for genes of interest
    annotation = pd.read_csv(local_path, 
                             sep='\t', 
                             header=None)
    annotation = extract(annotation, gene)
    
    # Store variant IDs and annotation records
    ids.append(annotation['var'])
    annotations.append(annotation)

# Combine variant IDs and annotations across chromosomes
ids = np.array(pd.concat(ids))
annotations = pd.concat(annotations)

# Identify loss-of-function variants
lof = annotations[annotations['anno'].str.contains('LOFTEE-HC|SpliceAI-0.8')]
# Identify damaging variants
missense = annotations[annotations['anno'].str.contains('AlphaMissense-0.564')]


In [ ]:
# Format Hail variant genotype QC data
def format(df): 
        '''
        Format Hail variant genotype QC data.
        df : Hail QC table
        '''
        df["alleles"] = df["alleles"].apply(ast.literal_eval)
        df["ref"] = df["alleles"].str[0]
        df["alt"] = df["alleles"].str[1]
        df['id'] = df['locus'] + ':' + df['ref'] + ':' + df['alt']
        df = df[['id', 's', 'DP', 'VAF', 'AB', 'GT', 'AD', 'GQ', 'PS', 'RGQ', 'FT']]
        return df 


# Define GCS path
gcs_path = f'{os.environ["WORKSPACE_BUCKET"]}/data/hail/'

# List files in GCS directory
result = subprocess.run(
    ["gcloud", "storage", "ls", gcs_path],
    check=True,
    capture_output=True,
    text=True
)

# Identify chromosome-specific genotype QC files
paths = result.stdout.strip().splitlines()
variant_paths = [
    path for path in paths
    if "variant_genotype_information_chr" in path
]

# Initialize QC data storage
qc = [] 

# Download and process each chromosome-specific file
for path in variant_paths:

    # Download QC file
    local_path = f"tmp.txt.gz"
    subprocess.run(
        ["gcloud", "storage", "cp", path, local_path],
        check=True
    )
    
    # Load and format genotype QC data
    df = pd.read_csv(
            local_path,
            sep='\t',
            compression='gzip'
        )
    df = format(df) 
    
    # Store formatted QC data
    qc.append(df) 

# Combine chromosome-specific QC data
qc = pd.concat(qc)

# Retain variants passing genotype-level QC
qc = qc[qc['FT'] =='PASS']

# Restrict to variants identified in the annotation analysis 
subset_qc = qc[qc['id'].isin(ids)]

# Apply allele balance (0.35-0.65) and read depth (>=20) filters
filtered_qc = subset_qc[~((subset_qc['AB'] < 0.35) | (subset_qc['AB'] > 0.65) | (subset_qc['DP'] < 20))]

In [ ]:
# Restrict QC datasets to LoF and missense variants
combined_subset_qc = subset_qc[subset_qc['id'].isin(pd.concat([lof, missense])['var'])]
combined_filtered_qc = filtered_qc[filtered_qc['id'].isin(pd.concat([lof, missense])['var'])]

combined_subset_qc = combined_subset_qc.copy()
combined_filtered_qc = combined_filtered_qc.copy()

# Map chromosomes to gene symbols
mapping = {
    "chr2": "DNMT3A",
    "chr4": "TET2",
    "chr15": "IDH2",
    "chr17": "SRSF2"
}

# Assign gene symbols based on variant chromosome
combined_subset_qc["symbol"] = (
    combined_subset_qc["id"]
    .str.extract(r"^(chr\d+)", expand=False)
    .map(mapping)
)

combined_filtered_qc["symbol"] = (
    combined_filtered_qc["id"]
    .str.extract(r"^(chr\d+)", expand=False)
    .map(mapping)
)

# Calculate genotype QC summary statistics before filtering
summary = combined_subset_qc.groupby('symbol').agg(
        Nsamples=('id', 'size'),
        meanAB=('AB', 'mean'),
        meanDP=('DP', 'mean'),
        meanVAF=('VAF', 'mean'),
        medianAB=('AB', 'median'),
        medianDP=('DP', 'median'),
        medianVAF=('VAF', 'median'),
        maximumAB=('AB', 'max'),
        maximumDP=('DP', 'max'),
        maximumVAF=('VAF', 'max'),
        minimumAB=('AB', 'min'),
        minimumDP=('DP', 'min'),
        minimumVAF=('VAF', 'min')
)

# Retain genes with more than 40 observations
summary = summary[summary['Nsamples'] > 40]
# Save pre-filter QC summary
summary.to_csv('QC_pre_filter_summary.tsv', sep='\t')

# Calculate genotype QC summary statistics after filtering
summary = combined_filtered_qc.groupby('symbol').agg(
        Nsamples=('id', 'size'),
        meanAB=('AB', 'mean'),
        meanDP=('DP', 'mean'),
        meanVAF=('VAF', 'mean'),
        medianAB=('AB', 'median'),
        medianDP=('DP', 'median'),
        medianVAF=('VAF', 'median'),
        maximumAB=('AB', 'max'),
        maximumDP=('DP', 'max'),
        maximumVAF=('VAF', 'max'),
        minimumAB=('AB', 'min'),
        minimumDP=('DP', 'min'),
        minimumVAF=('VAF', 'min')
)

# Retain genes with more than 40 observations
summary = summary[summary['Nsamples'] > 40]
# Save post-filter QC summary
summary.to_csv('QC_post_filter_summary.tsv', sep='\t')

In [ ]:
# Calculate variant-level QC summary statistics before filtering
summary = combined_subset_qc.groupby('id').agg(
        Nsamples=('id', 'size'),
        meanAB=('AB', 'mean'),
        meanDP=('DP', 'mean'),
        meanVAF=('VAF', 'mean'),
        medianAB=('AB', 'median'),
        medianDP=('DP', 'median'),
        medianVAF=('VAF', 'median'),
        maximumAB=('AB', 'max'),
        maximumDP=('DP', 'max'),
        maximumVAF=('VAF', 'max'),
        minimumAB=('AB', 'min'),
        minimumDP=('DP', 'min'),
        minimumVAF=('VAF', 'min')
)

# Retain variants observed in more than 40 samples
summary = summary[summary['Nsamples'] > 40]
# Save pre-filter variant-level QC summary
summary.to_csv('QC_pre_snv_summary.tsv', sep='\t')

# Calculate variant-level QC summary statistics after filtering
summary = combined_filtered_qc.groupby('id').agg(
        Nsamples=('id', 'size'),
        meanAB=('AB', 'mean'),
        meanDP=('DP', 'mean'),
        meanVAF=('VAF', 'mean'),
        medianAB=('AB', 'median'),
        medianDP=('DP', 'median'),
        medianVAF=('VAF', 'median'),
        maximumAB=('AB', 'max'),
        maximumDP=('DP', 'max'),
        maximumVAF=('VAF', 'max'),
        minimumAB=('AB', 'min'),
        minimumDP=('DP', 'min'),
        minimumVAF=('VAF', 'min')
)

# Retain variants observed in more than 40 samples
summary = summary[summary['Nsamples'] > 40]
# Save post-filter variant-level QC summary
summary.to_csv('QC_post_snv_summary.tsv', sep='\t')

# EXTRA PLOTS 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

combined_subset_qc = subset_qc[subset_qc['id'].isin(pd.concat([lof, missense])['var'])]
combined_filtered_qc = filtered_qc[filtered_qc['id'].isin(pd.concat([lof, missense])['var'])]

cols = ["AB", "DP", "VAF"]
names = ["Allele Balance", "Read Depth", "Variant Allele Fraction"]
chroms = ['chr2:', 'chr4:', 'chr15:', 'chr17:']
symbols = ['DNMT3A', 'TET2', 'IDH2', 'SRSF2']

for chrom, symbol in zip(chroms, symbols):

    plot_data1 = combined_subset_qc.reset_index(drop=True)
    plot_data2 = combined_filtered_qc.reset_index(drop=True)

    plot_data1 = plot_data1[plot_data1['id'].str.contains(chrom)]
    plot_data2 = plot_data2[plot_data2['id'].str.contains(chrom)]
    
    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 8)
    )
    

    for ax, col, name in zip(axes[0], cols, names):
        sns.histplot(
            data=plot_data1,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Unfiltered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    

    for ax, col, name in zip(axes[1], cols, names):
        sns.histplot(
            data=plot_data2,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Filtered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    
    plt.tight_layout()

    plt.savefig(f"QC_{symbol}_distributions.png", dpi=300, bbox_inches="tight")

    plt.savefig(f"QC_{symbol}_distributions.pdf", bbox_inches="tight")
    
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

lof_subset_qc = subset_qc[subset_qc['id'].isin(lof['var'])]
lof_filtered_qc = filtered_qc[filtered_qc['id'].isin(lof['var'])]

cols = ["AB", "DP", "VAF"]
names = ["Allele Balance", "Read Depth", "Variant Allele Fraction"]
chroms = ['chr2:', 'chr4:', 'chr15:', 'chr17:']
symbols = ['DNMT3A', 'TET2', 'IDH2', 'SRSF2']

for chrom, symbol in zip(chroms, symbols):

    plot_data1 = lof_subset_qc.reset_index(drop=True)
    plot_data2 = lof_filtered_qc.reset_index(drop=True)

    plot_data1 = plot_data1[plot_data1['id'].str.contains(chrom)]
    plot_data2 = plot_data2[plot_data2['id'].str.contains(chrom)]
    
    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 8)
    )
    
    for ax, col, name in zip(axes[0], cols, names):
        sns.histplot(
            data=plot_data1,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Pre-Filtered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    
    for ax, col, name in zip(axes[1], cols, names):
        sns.histplot(
            data=plot_data2,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Post-Filtered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    
    plt.tight_layout()

    plt.savefig(f"LoF_QC_{symbol}_distributions.png", dpi=300, bbox_inches="tight")
    
    plt.savefig(f"LoF_QC_{symbol}_distributions.pdf", bbox_inches="tight")
    
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

missense_subset_qc = subset_qc[subset_qc['id'].isin(missense['var'])]
missense_filtered_qc = filtered_qc[filtered_qc['id'].isin(missense['var'])]

cols = ["AB", "DP", "VAF"]
names = ["Allele Balance", "Read Depth", "Variant Allele Fraction"]
chroms = ['chr2:', 'chr4:', 'chr15:', 'chr17:']
symbols = ['DNMT3A', 'TET2', 'IDH2', 'SRSF2']

for chrom, symbol in zip(chroms, symbols):
    
    plot_data1 = missense_subset_qc.reset_index(drop=True)
    plot_data2 = missense_filtered_qc.reset_index(drop=True)

    plot_data1 = plot_data1[plot_data1['id'].str.contains(chrom)]
    plot_data2 = plot_data2[plot_data2['id'].str.contains(chrom)]
    
    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 8)
    )
    
    for ax, col, name in zip(axes[0], cols, names):
        sns.histplot(
            data=plot_data1,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Unfiltered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    
    for ax, col, name in zip(axes[1], cols, names):
        sns.histplot(
            data=plot_data2,
            x=col,
            bins=50,
            kde=True,
            ax=ax
        )
        
        ax.set_title(f"{symbol} - {name} - Filtered")
        ax.set_xlabel(f'{name} ({col})')
        ax.set_ylabel("Count")
    
    plt.tight_layout()

    plt.savefig(f"Missense_QC_{symbol}_distributions.png", dpi=300, bbox_inches="tight")
    
    plt.savefig(f"Missense_QC_{symbol}_distributions.pdf", bbox_inches="tight")
    
    plt.show()